# 3 - Nested cross-validation, niches and cross-fold metaclusters

Vignette 2 pretrained an encoder and fine-tuned one model on one split. This
notebook runs the preprint's **analysis chain** on the same tiles:

1. `grass-mil-nested-cv`: outer folds, inner epoch selection, refit, out-of-fold scores;
2. `grass-mil-extract`: every cell of every tile through every fold model;
3. `grass-mil-niches`: niches of one fold model with attention-weighted attribution;
4. `grass-mil-metaclusters`: which niches replicate across the fold models;
5. `grass-mil-scores`: bootstrap intervals and operating points of the out-of-fold scores;
6. `grass-mil-spatial-stats`: the model-free comparison.

---

### Read this before interpreting any number below

The dataset is three images from two donors, cut into ~40 tiles. The
**unit** of everything below is a tile (`id_kind=patch`): tiles are the bags,
the labelled units and the fold members. The label is the synthetic
epithelial-rich flag of vignette 2, a positive control. Tiles of one image are
grouped as one "patient" so that no image contributes to both training and
test of a fold, which leaves two outer folds (one image / two images) and one
image per inner fold. Everything runs in a few minutes on CPU with two epochs
per run, so the numbers show the mechanics and the outputs, not a result.
The preprint uses 5 outer folds, 3 inner folds, 25 epochs and whole cores as
units.


## Setup

In [ ]:
import json
import os
import pathlib
import subprocess
import sys
import glob

import numpy as np
import pandas as pd

WORK = pathlib.Path("vignette_work").resolve()
RAW, PROCESSED = WORK / "data" / "raw", WORK / "data" / "processed"
assert (RAW / "patch_labels.csv").exists(), "Run vignettes 1 and 2 first."
env = dict(os.environ, PROJECT_ROOT=str(WORK))

DATA_ARGS = [
    "data=spatial_omics",
    f"data.raw_manifest_path={RAW}/manifest.csv",
    f"data.processed_dir={PROCESSED}",
    "data.h5ad.coord_source=obsm",
    "data.h5ad.coord_key=spatial",
    "data.h5ad.categorical_label_columns=[cell_type]",
    "data.coord_scale_um=0.39",
    "data.sample_unit=tile",
    "data.tiling.tile_size_um=100",
    "data.tiling.stride_um=100",
    "data.min_cells=20",
    "data.tiling.min_cells=20",
    "data.graph_builder.kwargs.edge_features=[log_distance,neighbor]",
    "data.graph_builder.kwargs.neighbor_cutoff_um=20",
    "data.split.train_val_test_split=[0.34,0.33,0.33]",
    "data.split.split_by=sample",
]
EGO_ARGS = ["data.sampler.runtime.depth=2", "data.sampler.runtime.subgraph_batch_size=32"]
BAG_ARGS = [
    "data.sampler.runtime.subsample_count=32",
    "data.val_sampler.runtime.subsample_count=64",
    "task.region_accumulation.hyperbatch_size=4",
]
bgrl_ckpt = sorted(glob.glob(f"{WORK}/logs/pretrain_a3/runs/*/checkpoints/last.ckpt"), key=os.path.getmtime)[-1]
print("pretrained encoder:", os.path.relpath(bgrl_ckpt, WORK))


def run(args, tag):
    out = subprocess.run([sys.executable, "-m"] + args, capture_output=True, text=True, env=env)
    print(f"{tag}: exit {out.returncode}")
    if out.returncode:
        print(out.stdout[-3000:], out.stderr[-3000:])
        raise RuntimeError(f"{tag} failed - see output above")
    return out


## 1. Labels, patients and folds

The label file needs the unit id, the target and a patient column. Here a
"patient" is an image, so tiles of the same image never straddle a fold.
Outer folds are a JSON list of lists of unit ids.


In [ ]:
index = json.loads((PROCESSED / "processed_index.json").read_text())
entries = index["entries"] if "entries" in index else index
labels = pd.read_csv(RAW / "patch_labels.csv")
image_of = {e["patch_id"]: e["sample_id"] for e in entries}
labels["patient"] = labels["patch_id"].map(image_of)
labels = labels.dropna(subset=["patient"])
labels.to_csv(RAW / "tile_labels.csv", index=False)
print(labels.groupby("patient")["epithelial_rich"].agg(tiles="size", positive="sum").to_string())

# Two outer folds grouped by image: one image | the other two.
images = sorted(labels["patient"].unique())
folds = [
    labels.loc[labels.patient == images[0], "patch_id"].tolist(),
    labels.loc[labels.patient.isin(images[1:]), "patch_id"].tolist(),
]
(RAW / "folds.json").write_text(json.dumps(folds))
print("fold sizes:", [len(f) for f in folds])


## 2. Nested cross-validation

`regime=mil` is the GRASS-MIL default. For every outer fold the driver splits
the development tiles into `inner_k` image-grouped inner folds, trains with
region-level selection, takes the median best epoch, refits on all development
tiles for that many epochs and scores the held-out tiles once, on all cells.


In [ ]:
NESTED = WORK / "nested_mil"
run(
    [
        "grass_mil.nested_cv",
        "regime=mil",
        "trainer=cpu",
        "trainer.precision=32",
        f"model.init_from_ckpt={bgrl_ckpt}",
        *DATA_ARGS,
        *EGO_ARGS,
        *BAG_ARGS,
        f"data.graph_labels.label_file={RAW}/tile_labels.csv",
        "data.graph_labels.id_column=patch_id",
        "data.graph_labels.tasks=[epithelial_rich]",
        "data.graph_labels.patient_column=patient",
        "data.graph_labels.scope=patch",
        "data.force_precompute=true",
        "nested_cv.id_kind=patch",
        f"nested_cv.folds_file={RAW}/folds.json",
        "nested_cv.inner_k=2",
        "trainer.max_epochs=2",
        "trainer.min_epochs=1",
        f"paths.output_dir={NESTED}",
        f"paths.log_dir={WORK}/logs",
    ],
    "nested CV",
)
summary = json.loads((NESTED / "nested_cv_summary.json").read_text())
print({k: summary[k] for k in ("auc_mean", "auc_sd", "auc_folds", "patient_auroc", "complete")})
oof = pd.read_csv(NESTED / "oof_predictions.csv")
oof.head()


`oof_predictions.csv` has one held-out score per tile, `oof_patient_predictions.csv`
the mean per image. `outer_<i>/result.json` records the inner best epochs and the
refit length `E`.


## 3. Evaluation statistics

`grass-mil-scores` standardises the scores within fold (the two refit models
have different scales), reports the pooled AUROC with a bootstrap interval, the
per-fold mean, and operating points at pre-specified "flag the top q" rules. At
patient level there are only three images here, so we score the tiles
(`patient_level=false`).


In [ ]:
run(
    [
        "grass_mil.evaluation.cli",
        f"input={NESTED}/oof_predictions.csv",
        f"output_dir={WORK}/scores",
        "patient_level=false",
        "n_boot=500",
        f"hydra.run.dir={WORK}/scores/hydra",
    ],
    "scores",
)
print(pd.read_csv(f"{WORK}/scores/scores_summary.csv").T)
ops = pd.read_csv(f"{WORK}/scores/operating_points.csv")
ops[["rule", "flagged", "sensitivity", "specificity", "ppv", "npv"]].round(2)


## 4. Extract every cell through every fold model

The cache holds, per cell, the ego-graph embedding, the per-class instance
logits and the raw attention logits of each refit model, plus the cell types,
positions and the cell graph. The held-out tiles of every model are recorded.


In [ ]:
run(
    [
        "grass_mil.extract",
        "regime=mil",
        "trainer=cpu",
        *DATA_ARGS,
        *EGO_ARGS,
        f"data.graph_labels.label_file={RAW}/tile_labels.csv",
        "data.graph_labels.id_column=patch_id",
        "data.graph_labels.tasks=[epithelial_rich]",
        "data.graph_labels.patient_column=patient",
        "data.graph_labels.scope=patch",
        "task.bag_key=patch_id",
        f"extract.nested_cv_summary={NESTED}/nested_cv_summary.json",
        "extract.device=cpu",
        f"extract.output_dir={WORK}/cells",
        f"paths.output_dir={WORK}/extract",
    ],
    "extract",
)
for d in sorted(glob.glob(f"{WORK}/cells/*/index.json")):
    idx = json.loads(open(d).read())
    print(idx["model_name"], ":", len(idx["regions"]), "tiles,", len(idx["held_out_regions"]), "held out, unit =", idx["unit_id"])


## 5. Niches of one fold model

Cells of the tiles the model never trained on are clustered in its own
embedding space. The preprint's recipe is PCA-10 -> L2 -> HDBSCAN(300, 1, eom)
-> 15-NN assignment; with a few thousand cells the minimum cluster size is
lowered to 30 and no sampling is needed. Every cell gets the attention-weighted
margin `relattn`; every cluster gets its composition, empirical association
and margin with a bootstrap interval over tiles.


In [ ]:
run(
    [
        "grass_mil.interpretability.niches_cli",
        f"cells_dir={WORK}/cells/fold1",
        f"output_dir={WORK}/niches_fold1",
        "clustering.params.min_cluster_size=30",
        "sample_size=null",
        "n_boot=300",
        "positive_name=epithelial-rich",
        f"hydra.run.dir={WORK}/niches_fold1/hydra",
    ],
    "niches",
)
clusters = pd.read_csv(f"{WORK}/niches_fold1/clusters.csv")
clusters[["cluster", "share", "cores_present", "top_types", "emp_frac_pos", "share_auroc", "relattn", "relattn_lo", "relattn_hi"]].round(2)


In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{WORK}/niches_fold1/pca_triptych.png", width=900))
display(Image(filename=f"{WORK}/niches_fold1/clusters_overview.png", width=900))


Left: clusters in the first two PCs of the held-out cells. Middle: the
attention-weighted margin (red = pushes towards epithelial-rich). Right: the
empirical association (fraction of cells from positive tiles). The overview
shows composition, association and margin per cluster side by side; with a
positive control the epithelial clusters should carry positive margins.


## 6. Cross-fold metaclusters

Each fold model is clustered on its own held-out cells with the same settings;
clusters are matched across folds by Hellinger distance on their composition,
never merging two clusters of the same fold. A metacluster that has members from
`match.min_folds` folds replicates. Shares per tile (each tile labelled once, by
the model that held it out) are tested at the patient level; the transfer stage
labels every tile with every model and the basis stage fits one shared k-means
basis for all models.


In [ ]:
run(
    [
        "grass_mil.interpretability.metaclusters_cli",
        f"cells_dir={WORK}/cells",
        f"output_dir={WORK}/meta",
        f"labels.label_file={RAW}/tile_labels.csv",
        "labels.id_column=patch_id",
        "labels.label_column=epithelial_rich",
        "labels.patient_column=patient",
        "clustering.params.min_cluster_size=30",
        "sample_size=null",
        "n_boot=300",
        "match.min_folds=2",
        "transfer.min_cells=20",
        "basis.k=6",
        "basis.sample_size=5000",
        "basis.n_init=3",
        "positive_name=epithelial-rich",
        f"hydra.run.dir={WORK}/meta/hydra",
    ],
    "metaclusters",
)
meta = pd.read_csv(f"{WORK}/meta/match/metaclusters.csv")
meta[["meta", "n_folds", "share_mean", "top_types", "emp_rel_mean", "margin_mean", "sign_agree"]].round(2)


In [ ]:
display(Image(filename=f"{WORK}/meta/match/metaclusters_overview.png", width=900))
print(open(f"{WORK}/meta/transfer/report_transfer.md").read()[:1500])


## 7. Classical spatial statistics

The same cell cache feeds the model-free comparison: neighbourhood enrichment
on the cell graph, cellular neighbourhoods, contact fractions and cross-K,
each per tile, averaged per image and tested between classes. With three
images the patient-level tests cannot be significant; `min_per_group=1` lets
the tables fill for the demo.


In [ ]:
run(
    [
        "grass_mil.spatial_stats.cli",
        f"cells_dir={WORK}/cells/fold1",
        f"output_dir={WORK}/spatial",
        f"labels.label_file={RAW}/tile_labels.csv",
        "labels.id_column=patch_id",
        "labels.label_column=epithelial_rich",
        "labels.patient_column=patch_id",  # tiles as the test unit for the demo
        "reference=[Epithelial]",
        "partners=[Myeloid_CD68,Myeloid_CD11c,Tcell_CD8,Tcell_CD4,Fibroblast]",
        "infiltrating=[Myeloid_CD68,Tcell_CD8]",
        "nhood.n_perms=100",
        "cn.n_cn=4",
        "cn.sample_size=5000",
        "crossk.radii=[10,20,40]",
        "min_per_group=3",
        "positive_name=epithelial-rich",
        f"hydra.run.dir={WORK}/spatial/hydra",
    ],
    "spatial stats",
)
pd.read_csv(f"{WORK}/spatial/nhood_tests.csv").head(8).round(3)


## Where to go next

Point the same commands at your cohort: whole cores or regions as units
(`data.sample_unit=full`, `nested_cv.id_kind=region`), a real patient column,
patient-grouped folds, the pretraining preset at its defaults
(`task.total_steps=200000`, 4-hop ego-graphs) and the regimes of the preprint
(`regime=linear_probehead|linear_ftall|linear_ftall_reg`). See
`docs/nested_cv.md` and `docs/niche_discovery.md`.
